# Zero-Knowledge Proof Verification Walkthrough

## Quantum Circuit Resource Estimates for Breaking 256-bit ECDLP

This notebook walks through the mathematics behind the zero-knowledge proof described in:

> **"Securing Elliptic Curve Cryptocurrencies against Quantum Vulnerabilities: Resource Estimates and Mitigations"**
> Babbush, Zalcman, Gidney et al. — Google Quantum AI (2026)

The paper demonstrates that Shor's algorithm for breaking 256-bit ECDLP on the secp256k1 curve can be executed with dramatically fewer resources than previously published. A zero-knowledge proof validates these claims without revealing the secret circuits.

**What we cover:**
1. secp256k1 elliptic curve arithmetic
2. Fiat-Shamir fuzz testing and soundness analysis
3. Resource cost derivation (point addition → full ECDLP)
4. Physical resource estimation for superconducting architectures
5. Attack timing and success probability

## Part 1: The secp256k1 Elliptic Curve

The secp256k1 curve is defined by the equation:

$$y^2 = x^3 + 7 \pmod{p}$$

where $p = 2^{256} - 2^{32} - 977$ is a 256-bit prime.

**Point addition** on this curve is the core operation that the secret quantum circuit must compute correctly. Let's implement it and see how it works.

In [1]:
# secp256k1 curve parameters
p = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
a = 0
b = 7
# Generator point G
Gx = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798
Gy = 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8
# Group order
n = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141

print(f"Field prime p ({p.bit_length()} bits):")
print(f"  p = {p}")
print(f"\nGroup order n ({n.bit_length()} bits):")
print(f"  n = {n}")
print(f"\nGenerator G:")
print(f"  x = {hex(Gx)}")
print(f"  y = {hex(Gy)}")

# Verify G is on the curve
assert (Gy**2 - Gx**3 - 7) % p == 0, "G is not on the curve!"
print(f"\n✓ Generator G is on the secp256k1 curve")

Field prime p (256 bits):
  p = 115792089237316195423570985008687907853269984665640564039457584007908834671663

Group order n (256 bits):
  n = 115792089237316195423570985008687907852837564279074904382605163141518161494337

Generator G:
  x = 0x79be667ef9dcbbac55a06295ce870b07029bfcdb2dce28d959f2815b16f81798
  y = 0x483ada7726a3c4655da4fbfc0e1108a8fd17b448a68554199c47d08ffb10d4b8

✓ Generator G is on the secp256k1 curve


In [2]:
def modinv(a, m):
    """Modular inverse using extended Euclidean algorithm."""
    if a < 0:
        a = a % m
    g, x, _ = extended_gcd(a, m)
    if g != 1:
        raise ValueError("Modular inverse does not exist")
    return x % m

def extended_gcd(a, b):
    if a == 0:
        return b, 0, 1
    g, x, y = extended_gcd(b % a, a)
    return g, y - (b // a) * x, x

def point_add(P, Q, p):
    """Add two points on secp256k1 (y^2 = x^3 + 7 mod p)."""
    if P is None:
        return Q
    if Q is None:
        return P
    x1, y1 = P
    x2, y2 = Q
    if x1 == x2 and y1 != y2:
        return None  # Point at infinity
    if x1 == x2 and y1 == y2:
        # Point doubling
        lam = (3 * x1 * x1) * modinv(2 * y1, p) % p
    else:
        # Point addition
        lam = (y2 - y1) * modinv(x2 - x1, p) % p
    x3 = (lam * lam - x1 - x2) % p
    y3 = (lam * (x1 - x3) - y1) % p
    return (x3, y3)

def scalar_mult(k, P, p):
    """Multiply point P by scalar k using double-and-add."""
    result = None
    addend = P
    while k:
        if k & 1:
            result = point_add(result, addend, p)
        addend = point_add(addend, addend, p)
        k >>= 1
    return result

# Demonstrate: compute 2G, 3G, and verify
G = (Gx, Gy)
G2 = point_add(G, G, p)
G3 = point_add(G2, G, p)

print("Point addition on secp256k1:")
print(f"\n2G = G + G:")
print(f"  x = {hex(G2[0])}")
print(f"  y = {hex(G2[1])}")

print(f"\n3G = 2G + G:")
print(f"  x = {hex(G3[0])}")
print(f"  y = {hex(G3[1])}")

# Verify 3G via scalar multiplication
G3_check = scalar_mult(3, G, p)
assert G3 == G3_check
print(f"\n✓ 3G computed via addition matches scalar multiplication")

Point addition on secp256k1:

2G = G + G:
  x = 0xc6047f9441ed7d6d3045406e95c07cd85c778e4b8cef3ca7abac09b95c709ee5
  y = 0x1ae168fea63dc339a3c58419466ceaeef7f632653266d0e1236431a950cfe52a

3G = 2G + G:
  x = 0xf9308a019258c31049344f85f89d5229b531c845836f99b08601f113bce036f9
  y = 0x388f7b0f632de8140fe337e62a37f3566500a99934c2231b6cb9fd7584b8e672

✓ 3G computed via addition matches scalar multiplication


## Part 2: The ECDLP — Why It Matters

The **Elliptic Curve Discrete Logarithm Problem (ECDLP)** is: given points $G$ and $Q = kG$ on the curve, find the scalar $k$.

- Classically, this is believed to require $O(\sqrt{n})$ operations (best known: Pollard's rho algorithm)
- For secp256k1 with a 256-bit group order, this means ~$2^{128}$ operations — computationally infeasible
- **Shor's algorithm** solves ECDLP in polynomial time on a quantum computer

The private key $k$ is what secures cryptocurrency wallets. If you can solve ECDLP, you can derive the private key from any public key and forge transactions.

In [3]:
import secrets

# Demonstrate the ECDLP setup
# Generate a random "private key"
private_key = secrets.randbelow(n - 1) + 1

# Compute the "public key" Q = k * G
public_key = scalar_mult(private_key, G, p)

print("ECDLP Demonstration")
print("=" * 60)
print(f"\nPrivate key k (SECRET):  {hex(private_key)}")
print(f"\nPublic key Q = k * G (PUBLIC):")
print(f"  Q.x = {hex(public_key[0])}")
print(f"  Q.y = {hex(public_key[1])}")

# Verify the public key is on the curve
x, y = public_key
assert (y**2 - x**3 - 7) % p == 0
print(f"\n✓ Public key is on secp256k1")

print(f"\nThe ECDLP challenge: given G and Q, find k.")
print(f"Classical difficulty: ~2^128 operations")
print(f"Quantum (Shor's):    polynomial in 256 (the bit length)")

ECDLP Demonstration

Private key k (SECRET):  0x27837a76b5941ef121f24769657403233f86e12849f19e9239778985e2be6f1b

Public key Q = k * G (PUBLIC):
  Q.x = 0x850e76f091b0c1df9831c423e96129001d80ce80bc51c20274ccc493394ceb9c
  Q.y = 0x960204ee68703b26cdea488a31ada1325562e2090442f44439067e74399643b7

✓ Public key is on secp256k1

The ECDLP challenge: given G and Q, find k.
Classical difficulty: ~2^128 operations
Quantum (Shor's):    polynomial in 256 (the bit length)


## Part 3: The Fiat-Shamir Fuzz Testing Framework

The zero-knowledge proof doesn't reveal the secret circuit — instead it proves the circuit is **approximately correct** on random inputs. Here's how:

### Why approximate correctness is sufficient

Shor's algorithm works by creating a quantum superposition and using interference to extract the discrete logarithm. If a small fraction of values in the superposition are wrong, the algorithm simply fails with that small probability. A circuit correct on 99% of inputs makes Shor's algorithm succeed at least 99% of the time.

### The Fiat-Shamir heuristic

To prevent the prover from cherry-picking easy inputs:
1. Hash the secret circuit with SHA-256 to get a unique fingerprint
2. Use that hash to seed SHAKE256 (a cryptographic extendable-output function)
3. Generate all test inputs deterministically from this seed

This ensures the test inputs are **committed to by the circuit itself** — you can't change the circuit without changing the tests.

In [4]:
import hashlib
import math

# --- Soundness Analysis ---
# If the circuit is wrong on a fraction `epsilon` of inputs,
# what is the probability it passes `t` random tests?

epsilon = 0.01   # 1% error rate
t = 9024         # number of test cases used in the paper

prob_all_pass = (1 - epsilon) ** t
log2_prob = t * math.log2(1 - epsilon)

print("Fiat-Shamir Fuzz Testing: Soundness Analysis")
print("=" * 60)
print(f"\nAssumed circuit error rate:  {epsilon*100:.0f}%")
print(f"Number of random tests:     {t:,}")
print(f"\nPr[all {t:,} tests pass | ≥1% errors]:")
print(f"  = (1 - {epsilon})^{t}")
print(f"  = (0.99)^{t}")
print(f"  ≈ 2^({log2_prob:.1f})")
print(f"  ≈ {prob_all_pass:.2e}")
print(f"\nThis provides {abs(log2_prob):.0f} bits of security — well above")
print(f"the 128-bit threshold for cryptographic confidence.")

print(f"\n--- Why 9,024 specifically? ---")
# Solve: (1 - 0.01)^t <= 2^(-128)
t_min = math.ceil(-128 / math.log2(1 - 0.01))
print(f"Minimum tests for 128-bit security at 1% error:")
print(f"  t ≥ ⌈-128 / log₂(0.99)⌉ = {t_min}")
print(f"  Paper uses {t} tests (provides {abs(log2_prob):.0f}-bit security)")

Fiat-Shamir Fuzz Testing: Soundness Analysis

Assumed circuit error rate:  1%
Number of random tests:     9,024

Pr[all 9,024 tests pass | ≥1% errors]:
  = (1 - 0.01)^9024
  = (0.99)^9024
  ≈ 2^(-130.8)
  ≈ 4.09e-40

This provides 131 bits of security — well above
the 128-bit threshold for cryptographic confidence.

--- Why 9,024 specifically? ---
Minimum tests for 128-bit security at 1% error:
  t ≥ ⌈-128 / log₂(0.99)⌉ = 8828
  Paper uses 9024 tests (provides 131-bit security)


In [5]:
# Demonstrate how Fiat-Shamir test generation works
# (We don't have the actual secret circuit, but we can show the process)

def generate_fiat_shamir_test_points(circuit_hash_hex, num_tests, curve_order, generator, field_prime):
    """
    Generate test points using the Fiat-Shamir heuristic.

    1. Seed SHAKE256 with the circuit hash bytes
    2. Sample random 256-bit scalars k
    3. Compute test points as G * k
    """
    # Initialize SHAKE256 with circuit hash bytes
    circuit_bytes = bytes.fromhex(circuit_hash_hex.replace("0x", ""))
    xof = hashlib.shake_256(circuit_bytes)

    # We need 32 bytes per scalar
    random_bytes = xof.digest(32 * num_tests)

    test_scalars = []
    for i in range(num_tests):
        k_bytes = random_bytes[i*32 : (i+1)*32]
        k = int.from_bytes(k_bytes, 'big') % curve_order
        if k == 0:
            k = 1  # avoid point at infinity
        test_scalars.append(k)

    return test_scalars

# Use the actual circuit hash from the paper (low-qubit variant)
low_qubit_hash = "0xcc8f532ffea1583ceed3c9af75de3263ebaddd5fdf3cddfb3dea848b94d0396a"

# Generate the first few test scalars
test_scalars = generate_fiat_shamir_test_points(low_qubit_hash, 5, n, G, p)

print("Fiat-Shamir Test Generation (Low-Qubit Circuit)")
print("=" * 60)
print(f"\nCircuit SHA-256: {low_qubit_hash}")
print(f"\nFirst 5 test scalars (derived from circuit hash via SHAKE256):")
for i, k in enumerate(test_scalars):
    print(f"  k[{i}] = {hex(k)}")

# Compute corresponding test points
print(f"\nFirst 3 test points Q = G * k:")
for i in range(3):
    Q = scalar_mult(test_scalars[i], G, p)
    print(f"  Q[{i}].x = {hex(Q[0])}")
    print(f"  Q[{i}].y = {hex(Q[1])}")
    # Verify on curve
    assert (Q[1]**2 - Q[0]**3 - 7) % p == 0
    print(f"  ✓ on curve")

Fiat-Shamir Test Generation (Low-Qubit Circuit)

Circuit SHA-256: 0xcc8f532ffea1583ceed3c9af75de3263ebaddd5fdf3cddfb3dea848b94d0396a

First 5 test scalars (derived from circuit hash via SHAKE256):
  k[0] = 0x1ede1fc0418fccffddd2b38e44e33a0c0d8d255f651745c48aa2473f9fa910a2
  k[1] = 0xfcca4864ad4fd90b531469e58d706ba4c91fbc15ae6327fed8cf2bd78659f4fe
  k[2] = 0x735f43b7b1995f2282d6d82b3417c2e6f6a0580d6d90de5e2f8f0a653172087
  k[3] = 0x15c48eb152d84e79ff26aa285aa7c394214cbb834dd1a1687ea08b190f05744b
  k[4] = 0x40ff621d9ff1cda55ab00c24f4f3f92690ffa7b5a0e2530a6be8f3b7704b6bfa

First 3 test points Q = G * k:
  Q[0].x = 0x743008eaa821724cfe683d0692258329e7c66c6808a3bb0d1426c2b051b2bd6d
  Q[0].y = 0xe5e6dcd0ec9c576d7bc04fcc505dbc237cf2842e11394908438feeed3564ccb3
  ✓ on curve
  Q[1].x = 0xd25c541e12bca5fed490f9e9ba625f18b2c3814998bc1d98165319ba8d1c7d07
  Q[1].y = 0xd1828f3a128f31277c12e688f698e55aabae213b943efe10e84dc0d74bc90d4a
  ✓ on curve
  Q[2].x = 0x29badd678ae97e6cf58137137ba6e18e52840cb8f

## Part 4: From Point Addition to Full ECDLP — Resource Cost Derivation

The ZK proof directly attests to the resources of the **point addition subroutine**. The full ECDLP algorithm cost follows from well-known formulas.

### Shor's Algorithm Structure for ECDLP

Shor's algorithm for ECDLP uses **phase estimation across two variables**, each requiring $n = 256$ controlled elliptic curve point additions. Using **windowed arithmetic** with window size $w$:

$$\text{ECDLP}_{\text{Toff}} = \left(\text{PA}_{\text{Toff}} + 3 \times 2^w\right) \times \left(\frac{2n}{w} - 4\right)$$

$$\text{ECDLP}_{\text{Qubits}} = \text{PA}_{\text{Qubits}} + w$$

where $\text{PA}_{\text{Toff}}$ and $\text{PA}_{\text{Qubits}}$ are the point addition costs.

In [6]:
def ecdlp_resources(pa_toffoli, pa_qubits, n_bits=256, window_size=16):
    """
    Compute full ECDLP circuit resources from point addition costs.

    Parameters:
        pa_toffoli: Toffoli gates for one point addition
        pa_qubits: Logical qubits for one point addition
        n_bits: Bit length of the curve (256 for secp256k1)
        window_size: Window size for windowed arithmetic
    """
    num_windows = 2 * n_bits // window_size - 4
    window_lookup_cost = 3 * (2 ** window_size)
    total_toffoli = (pa_toffoli + window_lookup_cost) * num_windows
    total_qubits = pa_qubits + window_size
    return total_toffoli, total_qubits, num_windows

# --- Low-Qubit Variant ---
pa_toff_lq = 2_700_000   # from ZK proof statement 1
pa_qubits_lq = 1_175     # from ZK proof statement 1

toff_lq, qubits_lq, num_win = ecdlp_resources(pa_toff_lq, pa_qubits_lq)

print("Resource Derivation: Low-Qubit Variant")
print("=" * 60)
print(f"\nPoint Addition (from ZK proof):")
print(f"  Toffoli gates:  ≤ {pa_toff_lq:>12,}")
print(f"  Logical qubits: ≤ {pa_qubits_lq:>12,}")
print(f"\nWindow size w = 16")
print(f"Number of windowed point additions: 2×256/16 - 4 = {num_win}")
print(f"Window lookup cost: 3 × 2^16 = {3 * 2**16:,}")
print(f"\nFull ECDLP circuit:")
print(f"  Toffoli gates:  ({pa_toff_lq:,} + {3*2**16:,}) × {num_win} = {toff_lq:,}")
print(f"  Logical qubits: {pa_qubits_lq:,} + 16 = {qubits_lq:,}")
print(f"\n  → Paper reports: ≤ 90M Toffoli, ≤ 1,200 qubits ✓")

print()

# --- Low-Gate Variant ---
pa_toff_lg = 2_100_000   # from ZK proof statement 2
pa_qubits_lg = 1_425     # from ZK proof statement 2

toff_lg, qubits_lg, _ = ecdlp_resources(pa_toff_lg, pa_qubits_lg)

print("Resource Derivation: Low-Gate Variant")
print("=" * 60)
print(f"\nPoint Addition (from ZK proof):")
print(f"  Toffoli gates:  ≤ {pa_toff_lg:>12,}")
print(f"  Logical qubits: ≤ {pa_qubits_lg:>12,}")
print(f"\nFull ECDLP circuit:")
print(f"  Toffoli gates:  ({pa_toff_lg:,} + {3*2**16:,}) × {num_win} = {toff_lg:,}")
print(f"  Logical qubits: {pa_qubits_lg:,} + 16 = {qubits_lg:,}")
print(f"\n  → Paper reports: ≤ 70M Toffoli, ≤ 1,450 qubits ✓")

Resource Derivation: Low-Qubit Variant

Point Addition (from ZK proof):
  Toffoli gates:  ≤    2,700,000
  Logical qubits: ≤        1,175

Window size w = 16
Number of windowed point additions: 2×256/16 - 4 = 28
Window lookup cost: 3 × 2^16 = 196,608

Full ECDLP circuit:
  Toffoli gates:  (2,700,000 + 196,608) × 28 = 81,105,024
  Logical qubits: 1,175 + 16 = 1,191

  → Paper reports: ≤ 90M Toffoli, ≤ 1,200 qubits ✓

Resource Derivation: Low-Gate Variant

Point Addition (from ZK proof):
  Toffoli gates:  ≤    2,100,000
  Logical qubits: ≤        1,425

Full ECDLP circuit:
  Toffoli gates:  (2,100,000 + 196,608) × 28 = 64,305,024
  Logical qubits: 1,425 + 16 = 1,441

  → Paper reports: ≤ 70M Toffoli, ≤ 1,450 qubits ✓


## Part 5: Physical Resource Estimation

The logical circuit must be mapped onto physical hardware with quantum error correction. The paper assumes:

- **Architecture:** Superconducting qubits with planar degree-4 connectivity
- **Physical error rate:** $10^{-3}$ (consistent with Google's demonstrated hardware)
- **Error correction:** Surface code with "yoke"-based dense qubit storage
- **Execution mode:** Reaction-limited (speed limited by 10μs control system reaction time)

### Key physical resource calculations

In [7]:
print("Physical Resource Estimation")
print("=" * 60)

# --- Wall clock time estimation ---
reaction_time_us = 10       # microseconds (control system reaction time)
overhead_per_toffoli = 1.5  # 50% overhead per Toffoli gate

for label, toff_count in [("Low-Gate (70M)", 70_000_000), ("Low-Qubit (90M)", 90_000_000)]:
    time_us = toff_count * reaction_time_us * overhead_per_toffoli
    time_min = time_us / 1e6 / 60
    print(f"\n{label} variant:")
    print(f"  {toff_count/1e6:.0f}M Toffoli × {reaction_time_us}μs × {overhead_per_toffoli} = {time_us/1e6:.0f}s = {time_min:.0f} minutes")

# --- Primed attack time ---
print(f"\n--- 'Primed' On-Spend Attack ---")
print(f"The algorithm can precompute the first half (which depends only on")
print(f"protocol parameters). The attack time from the moment a public key")
print(f"is revealed is approximately HALF the full time:")
for label, toff_count in [("Low-Gate", 70_000_000), ("Low-Qubit", 90_000_000)]:
    time_us = toff_count * reaction_time_us * overhead_per_toffoli
    primed_min = time_us / 1e6 / 60 / 2
    print(f"  {label}: ~{primed_min:.0f} minutes")

# --- T state production ---
print(f"\n--- T State Production (Magic State Distillation) ---")
toff_per_sec_needed = 70_000_000 / (9 * 60)  # 70M Toffoli in 9 minutes
t_state_qubits = 50_000  # qubits per T state at required fidelity
ec_round_us = 1  # microsecond EC rounds for fast-clock

t_states_per_sec = toff_per_sec_needed  # ~1 T state per Toffoli
qubits_for_magic = int(t_states_per_sec * t_state_qubits * ec_round_us / 1e6)

print(f"  T states needed per second: ~{toff_per_sec_needed:,.0f}")
print(f"  Physical qubits per T state: ~{t_state_qubits:,} qubit-rounds")
print(f"  At 1μs EC rounds: ~{qubits_for_magic:,} physical qubits for magic production")
print(f"  (Small compared to ~500,000 total physical qubits)")

# --- Comparison to prior work ---
print(f"\n--- Improvement Over Prior Work ---")
prior_physical_qubits = 9_000_000  # Litinski 2023 (photonic architecture)
current_physical_qubits = 500_000

print(f"  Best prior estimate (Litinski '23): ~{prior_physical_qubits/1e6:.0f}M physical qubits")
print(f"  This work:                          ~{current_physical_qubits/1e3:.0f}K physical qubits")
print(f"  Reduction factor:                   ~{prior_physical_qubits/current_physical_qubits:.0f}x")

Physical Resource Estimation

Low-Gate (70M) variant:
  70M Toffoli × 10μs × 1.5 = 1050s = 18 minutes

Low-Qubit (90M) variant:
  90M Toffoli × 10μs × 1.5 = 1350s = 22 minutes

--- 'Primed' On-Spend Attack ---
The algorithm can precompute the first half (which depends only on
protocol parameters). The attack time from the moment a public key
is revealed is approximately HALF the full time:
  Low-Gate: ~9 minutes
  Low-Qubit: ~11 minutes

--- T State Production (Magic State Distillation) ---
  T states needed per second: ~129,630
  Physical qubits per T state: ~50,000 qubit-rounds
  At 1μs EC rounds: ~6,481 physical qubits for magic production
  (Small compared to ~500,000 total physical qubits)

--- Improvement Over Prior Work ---
  Best prior estimate (Litinski '23): ~9M physical qubits
  This work:                          ~500K physical qubits
  Reduction factor:                   ~18x


## Part 6: On-Spend Attack Success Probability

With a ~9 minute "primed" attack time on a superconducting CRQC, can an attacker break a key before a transaction is confirmed? Bitcoin's block time follows an exponential distribution with a mean of 10 minutes.

In [8]:
import numpy as np

def attack_success_probability(attack_time_min, mean_block_time_min):
    """
    Probability that a block takes longer than the attack time.
    Block mining is a Poisson process -> exponential inter-block times.

    P(block_time > attack_time) = exp(-attack_time / mean_block_time)
    """
    return np.exp(-attack_time_min / mean_block_time_min)

attack_time = 9  # minutes (primed superconducting CRQC estimate)

blockchains = {
    "Bitcoin":   10.0,    # minutes
    "Litecoin":   2.5,
    "Zcash":      1.25,   # 75 seconds
    "Dogecoin":   1.0,    # 60 seconds
}

print("On-Spend Attack Success Probability")
print("=" * 60)
print(f"\nAttack time (primed CRQC): {attack_time} minutes")
print(f"\n{'Blockchain':<12} {'Block Time':>12} {'P(success)':>14}")
print("-" * 40)

for chain, block_time in blockchains.items():
    prob = attack_success_probability(attack_time, block_time)
    if prob > 0.001:
        print(f"{chain:<12} {block_time:>10.1f}m {prob:>13.1%}")
    else:
        print(f"{chain:<12} {block_time:>10.1f}m {prob:>13.4%}")

print(f"\nBitcoin's ~41% risk means that roughly 2 in 5 transactions")
print(f"could be intercepted by a quantum attacker under ideal conditions.")
print(f"Shorter block times dramatically reduce this risk.")

On-Spend Attack Success Probability

Attack time (primed CRQC): 9 minutes

Blockchain     Block Time     P(success)
----------------------------------------
Bitcoin            10.0m         40.7%
Litecoin            2.5m          2.7%
Zcash               1.2m       0.0747%
Dogecoin            1.0m       0.0123%

Bitcoin's ~41% risk means that roughly 2 in 5 transactions
could be intercepted by a quantum attacker under ideal conditions.
Shorter block times dramatically reduce this risk.


In [9]:
# Visualize the race between attacker and blockchain

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 6))

    t_range = np.linspace(0, 45, 500)

    colors = {"Bitcoin": "#F7931A", "Litecoin": "#345D9D", "Zcash": "#ECB244", "Dogecoin": "#C2A633"}

    for chain, block_time in blockchains.items():
        # CDF of block time: P(block mined by time t) = 1 - exp(-t/mean)
        cdf = 1 - np.exp(-t_range / block_time)
        ax.plot(t_range, cdf * 100, label=f"{chain} ({block_time}m avg)",
                linewidth=2, color=colors[chain])

    # Attack completion line
    ax.axvline(x=attack_time, color='#FF4444', linestyle='--', linewidth=2,
               label=f'CRQC Key Derivation (~{attack_time}m)')

    # Shade the risk zone for Bitcoin
    btc_prob_at_attack = (1 - np.exp(-attack_time / 10.0)) * 100
    ax.fill_between([attack_time, 45], 0, 100, alpha=0.08, color='red')
    ax.annotate(f'~{100-btc_prob_at_attack:.0f}% Risk\n(Bitcoin block > {attack_time}m)',
                xy=(attack_time + 0.5, 100 - btc_prob_at_attack + 5),
                fontsize=11, color='#FF4444', fontweight='bold')

    ax.set_xlabel('Time Since Transaction Broadcast (minutes)', fontsize=12)
    ax.set_ylabel('Probability of Block Completion (%)', fontsize=12)
    ax.set_title('Race Against the Block: Attack Speed vs. Network Variance', fontsize=14)
    ax.legend(fontsize=10, loc='lower right')
    ax.set_xlim(0, 45)
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/home/user/quantum_proof/notebook/attack_timing.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("Plot saved to notebook/attack_timing.png")

except ImportError:
    print("matplotlib not available — skipping plot generation")
    print("Install with: pip install matplotlib")

Plot saved to notebook/attack_timing.png


## Part 7: The ZK Proof Verification Pipeline — Summary

The complete verification pipeline transforms the claim "we have a circuit with these resources" into a publicly verifiable cryptographic proof:

```
Secret Circuit C
       │
       ▼
┌──────────────────┐
│  SHA-256 Hash    │───▶ Public commitment (circuit fingerprint)
└──────────────────┘
       │
       ▼
┌──────────────────┐
│  SHAKE256 XOF    │───▶ 9,024 pseudo-random test inputs
│  (Fiat-Shamir)   │     (deterministically derived from circuit hash)
└──────────────────┘
       │
       ▼
┌──────────────────┐
│  Kickmix         │───▶ Simulate circuit on all test inputs
│  Simulator       │───▶ Count non-Clifford gates (CCX + CCZ)
│                  │───▶ Verify all outputs match expected point additions
└──────────────────┘
       │
       ▼
┌──────────────────┐
│  Assert Resource │───▶ Qubits ≤ bound, Toffoli gates ≤ bound
│  Bounds          │
└──────────────────┘
       │
       ▼
┌──────────────────┐
│  SP1 zkVM        │───▶ Arithmetize entire execution into STARK proof
│  (RISC-V)        │───▶ Recursive composition across shards
└──────────────────┘
       │
       ▼
┌──────────────────┐
│  Groth16 SNARK   │───▶ Final succinct zero-knowledge proof
│  Wrapper         │───▶ Publicly verifiable
└──────────────────┘
```

### What the proof guarantees:
1. The prover possesses a specific circuit (committed via SHA-256 hash)
2. That circuit correctly computes secp256k1 point addition on 9,024 random inputs (≥99% correctness with 128-bit security)
3. The circuit fits within the claimed resource bounds (qubits and gates)
4. None of the circuit's internal details are revealed

### What the proof does NOT guarantee:
- That the circuit implements the *optimal* approach
- That the physical resource estimates are achievable with current hardware
- That Shor's algorithm will work perfectly with this circuit (though 99% correctness is sufficient)

### A practical irony
The Groth16 SNARK itself relies on pairing-friendly elliptic curves — making it vulnerable to the very quantum attacks analyzed in the paper. A sufficiently large quantum computer could forge this proof. But since CRQCs don't exist yet, the proof's soundness holds today.

In [10]:
# Summary table of both circuit variants

print("=" * 72)
print("  SUMMARY: Zero-Knowledge Proof Statements")
print("=" * 72)

data = [
    ("", "Low-Qubit Variant", "Low-Gate Variant"),
    ("─" * 24, "─" * 20, "─" * 20),
    ("PA Non-Clifford Gates", "≤ 2,700,000", "≤ 2,100,000"),
    ("PA Logical Qubits", "≤ 1,175", "≤ 1,425"),
    ("PA Total Operations", "≤ 17,000,000", "≤ 17,000,000"),
    ("Fuzz Tests Passed", "9,024 / 9,024", "9,024 / 9,024"),
    ("─" * 24, "─" * 20, "─" * 20),
    ("ECDLP Toffoli Gates", "≤ 90 million", "≤ 70 million"),
    ("ECDLP Logical Qubits", "≤ 1,200", "≤ 1,450"),
    ("Physical Qubits", "< 500,000", "< 500,000"),
    ("Full Solve Time", "~23 minutes", "~18 minutes"),
    ("Primed Attack Time", "~12 minutes", "~9 minutes"),
]

for row in data:
    print(f"  {row[0]:<24} {row[1]:>20} {row[2]:>20}")

print(f"\n  Verification Key (shared):")
print(f"  0x00ca4af6cb15dbd83ec3eaab3a0664023828d90a98e650d2d340712f5f3eb0d4")
print(f"\n  Low-Qubit SHA-256:")
print(f"  0xcc8f532ffea1583ceed3c9af75de3263ebaddd5fdf3cddfb3dea848b94d0396a")
print(f"\n  Low-Gate SHA-256:")
print(f"  0x24f5758f2216aa87aa2806af32a0db788767b873cf6869510cca3d893b3f8a69")
print("=" * 72)

  SUMMARY: Zero-Knowledge Proof Statements
                              Low-Qubit Variant     Low-Gate Variant
  ──────────────────────── ──────────────────── ────────────────────
  PA Non-Clifford Gates             ≤ 2,700,000          ≤ 2,100,000
  PA Logical Qubits                     ≤ 1,175              ≤ 1,425
  PA Total Operations              ≤ 17,000,000         ≤ 17,000,000
  Fuzz Tests Passed               9,024 / 9,024        9,024 / 9,024
  ──────────────────────── ──────────────────── ────────────────────
  ECDLP Toffoli Gates              ≤ 90 million         ≤ 70 million
  ECDLP Logical Qubits                  ≤ 1,200              ≤ 1,450
  Physical Qubits                     < 500,000            < 500,000
  Full Solve Time                   ~23 minutes          ~18 minutes
  Primed Attack Time                ~12 minutes           ~9 minutes

  Verification Key (shared):
  0x00ca4af6cb15dbd83ec3eaab3a0664023828d90a98e650d2d340712f5f3eb0d4

  Low-Qubit SHA-256:
  0xcc8f

## References

- Babbush, Zalcman, Gidney et al. ["Securing Elliptic Curve Cryptocurrencies against Quantum Vulnerabilities: Resource Estimates and Mitigations"](https://quantumai.google/static/site-assets/downloads/cryptocurrency-whitepaper.pdf) (2026)
- Shor, P. "Algorithms for quantum computation: discrete logarithms and factoring" (1994)
- Groth, J. "On the Size of Pairing-based Non-interactive Arguments" (2016)
- Litinski, D. "How to compute a 256-bit elliptic curve private key with only 50 million Toffoli gates" (2023)
- Fiat, A. & Shamir, A. "How to prove yourself: Practical solutions to identification and signature problems" (1986)
- [SP1 zkVM — Succinct Labs](https://github.com/succinctlabs/sp1)

---
*This notebook is part of the [quantum_proof](https://github.com/chippr-robotics/quantum_proof) repository. It is intended as an educational resource and is not affiliated with Google Quantum AI.*